<a href='https://www.darshan.ac.in/'> <img src='https://www.darshan.ac.in/Content/media/DU_Logo.svg' width="250" height="300"/></a>
<pre>
<center><b><h1><strong>Data Mining</strong></h1></b></center>
<center><b><h1><strong>Lab - 10</strong></h1></b></center>
<center><b><h1><strong>¥@$# Kakadiya | 23010101121 | 20/8/2025</strong></h1></b></center>
<pre>



# Implement Decision Tree(ID3) in python
Uses Information Gain to choose the best feature to split.

Recursively builds the tree until stopping conditions are met.

1) Calculate Entropy for the dataset.<BR>
2) Calculate Information Gain for each feature. <BR>
3) Choose the feature with maximum Information Gain. <BR>
4) Split dataset into subsets for that feature. <BR>
5) Repeat recursively until: <BR>

All samples in a node have the same label.<BR>
No features are left.<BR>
No data is left.

### Step 2. Import the dataset from this [address](https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv).

##  import Pandas, Numpy

In [19]:
import pandas as pd
import numpy as np

##  Create Following Data

In [20]:
data = pd.DataFrame({
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
    'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
})

In [21]:
data

,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


## Now Define Function to  Calculate Entropy

In [22]:
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy = -np.sum([(counts[i]/np.sum(counts))*np.log2(counts[i]/np.sum(counts)) for i in range(len(elements))])
    return entropy

## Testing of Above Function -
y = np.array(['Yes', 'No',   'Yes', 'Yes'])  
Function Call - > entropy(y))

output - 0.8112781244591328

In [23]:
y = np.array(['Yes', 'No',   'Yes', 'Yes'])
print(entropy(y))

0.8112781244591328


## Define function to Calculate Information Gain

In [24]:
def information_gain(data, split_attribute_name, target_name):
    """
    Calculate the information gain of a dataset for a given attribute.
    """
    # Calculate the entropy of the whole dataset
    total_entropy = entropy(data[target_name])

    # Calculate the entropy of the dataset after splitting by the attribute
    values, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = np.sum([(counts[i]/np.sum(counts))*entropy(data.where(data[split_attribute_name]==values[i]).dropna()[target_name]) for i in range(len(values))])

    # Calculate the information gain
    information_gain = total_entropy - weighted_entropy
    return information_gain

## Testing of Above Function-
data = pd.DataFrame({
    'Weather': ['Sunny', 'Sunny', 'Rain', 'Rain'],
    'Play':    ['Yes', 'No',   'Yes', 'Yes']
})

Function Call - > information_gain(data, 'Weather', 'Play')


Output - 0.31127812445913283

In [25]:
data_test = pd.DataFrame({
'Weather': ['Sunny', 'Sunny', 'Rain', 'Rain'],
'Play':    ['Yes', 'No',   'Yes', 'Yes']
})

print(information_gain(data_test, 'Weather', 'Play'))

0.31127812445913283


## Implement ID3 Algo

In [26]:
def id3(data, features, target="PlayTennis"):
    # If all labels are same → return the label
    if len(np.unique(data[target])) == 1:
        return np.unique(data[target])[0]

    # If no features left → return majority label
    elif len(features) == 0:
        return np.unique(data[target])[np.argmax(np.unique(data[target], return_counts=True)[1])]

    else:
        # Choose best feature
        item_values = [information_gain(data, feature, target) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create tree structure
        tree = {best_feature: {}}

        # Remove best feature from feature list
        features = [i for i in features if i != best_feature]

        # For each value of best feature → branch
        for value in np.unique(data[best_feature]):
            sub_data = data.where(data[best_feature] == value).dropna()
            subtree = id3(sub_data, features, target)
            tree[best_feature][value] = subtree

        return tree

## Use ID3

In [27]:
features = data.columns[:-1]  # Exclude the target column
tree = id3(data, features)

## Print Tree

In [28]:
print(tree)

{'Outlook': {'Overcast': 'Yes', 'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}}, 'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}


## Extra: Create Predict Function

In [29]:
def predict(tree, sample):
    # If the node is a leaf node, return the label
    if not isinstance(tree, dict):
        return tree

    # Get the attribute to split on
    attribute = list(tree.keys())[0]
    attribute_value = sample[attribute]

    # Traverse the tree based on the attribute value
    if attribute_value in tree[attribute]:
        return predict(tree[attribute][attribute_value], sample)
    else:
        # Handle cases where the attribute value is not in the tree (e.g., return majority class of the current node's data, or a default value)
        # For simplicity, this implementation will return None or raise an error.
        # A more robust implementation would handle unknown values.
        return None # Or raise an error

##  Extra: Predict for a sample
sample = {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}


Your Answer ?

In [ ]:
sample = {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}

In [31]:
prediction = predict(tree, sample)
print("Prediction:", prediction)

Prediction: No


# Task
Complete the Python notebook to implement and test the ID3 decision tree algorithm, including functions for entropy, information gain, ID3 tree building, and prediction.

## Complete `entropy` function

### Subtask:
Implement the calculation of entropy in the provided `entropy` function.


**Reasoning**:
Implement the entropy function based on the provided instructions to calculate the entropy of a given target variable.



## Complete `information_gain` function

### Subtask:
Implement the calculation of information gain in the provided `information_gain` function.